<img src='logos/EUMETSAT_Logo_WhiteonBlue.png' alt='logo EUMETSAT' align='right' width='30%'></img>

## <span style="color:blue">**MTG LI flash geometry on FCI**</span>

<hr>

## Explore & Display MTG LI Flash Geometry on top of FCI imagery

<div class="alert alert-block alert-warning">
    
<b>PREREQUISITES </b>
    
This notebook has the following requirements:
    
- an MTG LI-2-LGR (Group) product, downloadable as described in the notebook [**Link to notebook on GitHub**](https://gitlab.eumetsat.int/eumetlab/data-services/eumdac_data_store/-/blob/master/1_6_MTG_LI_data_access.ipynb?ref_type=heads)
- an MTG FCI L1c product covering the same time/area, for the background image
    

</div>
<hr>

### About

This notebook builds on the companion notebook **LI_FlashData_Explore.ipynb**, which introduces the Level 2 Flash (LFL) product. Here, we instead work with the Level 2 **Group** (LGR) product, which retains the individual LI groups (the optical pulses detected within a single 1 ms acquisition frame) and the `flash_id` that links each group to its parent flash.<br>

By connecting the groups belonging to each flash, we obtain a proxy for the **geometry of the flash as seen from space** - the branching outline that the flash covers on the cloud top.<br>

Reference: <a href="https://www.essl.org/cms/mtg-li-at-essl-eumetsat-forecaster-testbeds/">ESSL: MTG LI at ESSL-EUMETSAT Forecaster Testbeds</a>, <a href="https://user.eumetsat.int/resources/user-guides/mtg-li-level-2-data-guide#ID-Group-and-flash-data">MTG LI Level 2 data guide</a>.

### Outline
This notebook is focusing on **LI flash geometry and group density displayed on top of FCI imagery**. It gives guidance on:

1) how to load LI-2-LGR (group) data and inspect its contents <br>
2) how to connect the groups belonging to each flash into a flash geometry, coloured by 2-minute time bin, together with a smoothed LI group density heatmap <br>
3) how to load and resample an FCI channel/composite to use as background imagery <br>
4) how to overlay the flash geometry and density on top of the FCI background for a given date/time and Region Of Interest

### Data used

| Product Description  | Data Store collection ID| Product Navigator | 
|:--------------------:|:-----------------------:|:-------------:|
| MTG LI level-2 Group | EO:EUM:DAT:0782         | <a href="https://data.eumetsat.int/data/map/EO:EUM:DAT:0688">link</a> |
| MTG FCI level-1c     | EO:EUM:DAT:0662         | <a href="https://data.eumetsat.int/data/map/EO:EUM:DAT:0662">link</a> |

A product is made of the following files:
* **Body chunks** (NetCDF): One body chunk containing the measured data values in NetCDF structure.
* Trailer chunk (NetCDF): One trailer chunk that contains at least the list of preceding body chunks that have been produced for the dataset. 
* Quicklooks (PNG): Images related to data that enable fast viewing.
* manifest.xml: The manifest contains further information about the SIP.
* EOPMetadata.xml: The EOPMetadata file contains metadata about the product.

### Module Content:
- [1 - Load and Inspect MTG LI data (GROUPS)](#LI_load)
- [2 - Flash geometry only](#Geometry_only)
- [3 - Flash geometry on FCI background](#Geometry_on_fci)
- [3.1 - Select time range](#Select_time)
- [3.2 - Select Region Of Interest](#Select_region)


<hr>

##### Load required libraries

We begin by importing all of the libraries that we need to run this notebook. If you have built your python using the environment file provided in this repository, then you should have everything you need. For more information on building the environment, please see the repository.

In [ ]:
from glob import glob
from satpy.scene import Scene
from AuxiliaryFunctions import plot_flash_geometry_density_from_filenames
from AuxiliaryFunctions import plot_flash_geometry_density_on_ir
from AuxiliaryFunctions import filter_filenames
from datetime import datetime
import warnings
warnings.filterwarnings("ignore")

## <a id='LI_load'></a>1. Load and Inspect LI data (Groups)

From the [EUMETSAT Data Store](https://data.eumetsat.int/data/map/EO:EUM:DAT:0688), we downloaded LI **Group** (LGR) data sensed for a specific day. The **LGR_Data_folder** variable should point to the folder containing LI LGR products, unzipped by the LI data download Notebook. As the data is contained in the **BODY** files, and since the data files have 'BODY' in their filenames, we will use the wildcard *BODY* to search for the relevant files within the folder. Another wildcard `/*/` is needed as each product is unzipped into its own subdirectory.

In [ ]:
LGR_Data_folder='downloads/*LI-2-LGR*BODY*.nc'


We save the list with the filenames in the folder into a variable called filenames, and sort the list:

In [ ]:
lgr_filenames = glob(LGR_Data_folder)
lgr_filenames.sort()

To access a file for a specific date, you can specify its position in the list with a specific index, e.g. `[2]` (indexing starts from 0).

In [ ]:
first_lgr_file = [lgr_filenames[0]]
first_lgr_file

As in the companion notebook, we use the [satpy](https://satpy.readthedocs.io/en/stable/) `li_l2_nc` reader to load the data. Once loaded, a `Scene` object represents a single geographic region of data, typically at a single continuous time range.

In [ ]:
scn = Scene(reader="li_l2_nc", 
            filenames=first_lgr_file)

With the function `available_dataset_names()`, you can see the datasets available in the LI Group data file. For flash geometry, the key datasets are `latitude`, `longitude` (per-group location) and `flash_id` (the identifier of the parent flash that each group belongs to).

In [ ]:
scn.available_dataset_names()

In [ ]:
latitude_dataset_name='latitude'
longitude_dataset_name='longitude'
flash_id_dataset_name='flash_id'
group_time_dataset_name='group_time'
scn.load([latitude_dataset_name, longitude_dataset_name, flash_id_dataset_name, group_time_dataset_name])
latitude_dataset = scn[latitude_dataset_name].compute()
longitude_dataset = scn[longitude_dataset_name].compute()
flash_id_dataset = scn[flash_id_dataset_name].compute()
group_time_dataset = scn[group_time_dataset_name].compute()

We can check how many individual groups, and how many distinct flashes, are contained in this single file:

In [ ]:
print('Number of groups: ', flash_id_dataset.size)
print('Number of distinct flashes: ', len(set(flash_id_dataset.values)))

## <a id='Geometry_only'></a>2. Flash geometry only

Before adding the FCI background, we first display the flash geometry and group density on their own. All the groups sharing the same `flash_id` are connected, and each geometry line segment is coloured according to the 2-minute time bin its group falls into (there are 5 2-minute bins inside 10 minute observation window) - so the time evolution of the storm is visible at a glance. This is layered under a smoothed heatmap of LI group density (groups per km$^2$).

In [ ]:
area_nswe = [40, 35, 0, 5]
title = 'LI flash geometry and group density (groups connected per flash, coloured by time bin)'

fig, ax = plot_flash_geometry_density_from_filenames(first_lgr_file, area_nswe=area_nswe, title=title)

## <a id='Geometry_on_fci'></a>3. Flash geometry on FCI background

To put the flash geometry and density into context, we overlay it on top of an FCI background channel (an inverted-grayscale IR channel by default, e.g. 10.8 $\mu$m brightness temperature - or a multi-band composite such as true colour, if you set the `channel` argument accordingly). The **FCI_Data_folder** variable should point to the folder containing FCI L1c products (BODY chunks), covering the same area and a time close to the LI data being displayed.

In [ ]:
FCI_Data_folder='../../eumdac_data_store/downloads/*BODY*.nc'

In [ ]:
fci_filenames = glob(FCI_Data_folder)
fci_filenames.sort()
fci_filenames

### <a id='Select_time'></a>3.1 Select Time Range

As in the companion notebook, this notebook can be narrowed to a specific time range, in order to select the LGR files relevant for the time of interest.

Current Time Range:

* start_dt = 2026-06-30 14:00
* end_dt = 2026-06-30 14:10

How to Modify:
* Set a new starting time by providing new numeric values in the format: start_dt = datetime (year, month, day, hour, minute).
* Set a new ending time by providing new numeric values in the format: end_dt = datetime (year, month, day, hour, minute). <br>
   For example, to set the range of times to be displayed between start time September 1, 2024, at 16:30 and 18:30, you would write:<br>
   start_dt = datetime (2024, 9, 1, 16, 30).<br>
   end_dt = datetime (2024, 9, 1, 18, 30).

  Note: please be aware that in **minute you should choose 0 / 10 /20 /30 /40 /50**

<div class="alert alert-block alert-success">
<b>You may want to modify the next cell:</b><br />
</div>

In [ ]:
start_dt = datetime(2026, 10, 19, 14, 0)
end_dt = datetime(2026, 10, 19, 14, 10)

In [ ]:
filtered_lgr_files = filter_filenames(lgr_filenames, start_dt, end_dt)
filtered_fci_files = filter_filenames(fci_filenames, start_dt, end_dt)

### <a id='Select_region'></a>3.2 Select Region Of Interest

The notebook needs a Region Of Interest (ROI) to resample the FCI background to, in the format **[lat_max, lat_min, lon_min, lon_max]**. Unlike the flash-only display, a ROI is required here (rather than optional), since the FCI background has to be resampled onto a defined grid.

<div class="alert alert-block alert-success">
<b>You may want to modify the next cell:</b><br />
</div>

In [ ]:
area_nswe = [45, 38, 0, 10]

In the next cell, we set the title and call the function that loads the FCI background, resamples it to the selected ROI, and overlays the LI flash geometry on top of it. This may take a little longer to run than the flash-only plot, since it includes loading and resampling the FCI imagery.

<div class="alert alert-block alert-success">
<b>You may want to change the channel (composite) and title of the plot in the next cell:</b><br />
</div>

In [ ]:
channel = 'true_color'
title = 'LI Flash Geometry & Density on FCI %s ' %(channel) + '[' + start_dt.strftime("%Y-%m-%d") + ' (' + start_dt.strftime("%H:%M") + '-' + end_dt.strftime("%H:%M") + ') ' + ']'

fig, ax = plot_flash_geometry_density_on_ir(filtered_lgr_files, filtered_fci_files, area_nswe,
                                             channel=channel, t_start=start_dt, title=title)

You may want to save the image to a file (PNG format). You can do that by running the below cell. This will save the file into the folder plots/, but if you want it saved elsewhere, you can change the output_filename accordingly.

In [ ]:
output_filename = "plots/fci_" + channel + "_%s.png" %(start_dt.strftime("%Y%m%dT%H%M%SZ"))
fig.savefig(output_filename, dpi=300, bbox_inches='tight')

<div class="alert alert-info" role="alert">

Please try to analyze the obtained map, keeping the following questions in mind as part of your interpretation: 

How large can the geometry of an individual flash be, compared to the convective cell it originates from? 

Do areas with higher LI group density (more groups per km$^2$) tend to occur over the most active parts of the convective cell, or can they extend well beyond it? 

How does the flash geometry relate to the cloud-top features visible in the FCI background image (e.g. overshooting tops, anvil edges)? 

Looking at the time-bin colours of the geometry lines, can you trace how the storm's lightning activity evolved or moved over the displayed period? 

What could explain cases where a flash's geometry appears to extend into apparently clear-sky areas in the FCI image? 

</div>

<details>
<summary><b>Q1: How large can the geometry of an individual flash be, compared to the convective cell it originates from?</b></summary>
<br>
A single LI flash can span anywhere from a single illuminated pixel (~4.5 km at nadir, larger toward the limb) up to tens of kilometres for long, horizontally-propagating flashes. Flash geometry is therefore frequently <em>larger</em> than the parent convective cell's core updraft region, especially once a storm matures and develops a broad stratiform anvil.
</details>

---

<details>
<summary><b>Q2: Do areas with higher LI group density tend to occur over the most active parts of the convective cell, or can they extend well beyond it?</b></summary>
<br>
High group density is a proxy for where flashes are repeatedly initiating - typically over the strongest updraft regions, which correlates well with the overshooting cloud tops. However, density can be elevated also in areas where many long flashes happen propagating. </details>

---

<details>
<summary><b>Q3: How does the flash geometry relate to cloud-top features visible in the FCI background image (e.g. overshooting tops, anvil edges)?</b></summary>
<br>
Densest group clusters typically sit over overshooting tops (the coldest, most textured IR pixels). 
</details>

---

<details>
<summary><b>Q4: Looking at the time-bin colours of the geometry lines, can you trace how the storm's lightning activity evolved or moved over the displayed period?</b></summary>
<br>
If the geometry colour progression (earlier bin to later bin) shifts spatially across the image, that oftentimes correlates well with the propagation direction of new convective initiation.
</details>

---

<details>
<summary><b>Q5: What could explain cases where a flash's geometry appears to extend into apparently clear-sky areas in the FCI image?</b></summary>
<br>
A few possible explanations: thin cloud that is invisible to the IR imager but still detectable by LI; parallax shift between LI and FCI (tall clouds appear displaced toward the limb); genuine out-of-cloud lightning travelling beyond the visible cloud edge; or small misalignments between the LI and FCI grids introduced during resampling.
</details>

<br>

<br>

<hr>

  
<p style="text-align:right;"> <a href='https://training.eumetsat.int'>EUMETSAT Training</a> | <a href='mailto:training@eumetsat.int'>Contact the training team</a></p>